#### Mediallion Architecture [!https://www.databricks.com/sites/default/files/inline-images/building-data-pipelines-with-delta-lake-120823.png]

In [0]:
dbutils.fs.ls('/FileStore/shared_uploads/kumar@vcloudmatesolutions.com')

### Brone Layer

In [0]:
dbutils.fs.mkdirs('/mydir/raw/')

In [0]:
display(dbutils.fs.ls('/mydir/raw'))

In [0]:
dbutils.fs.cp('dbfs:/FileStore/shared_uploads/kumar@vcloudmatesolutions.com/emp.csv','/mydir/raw/emp.csv')

In [0]:
dbutils.fs.cp('dbfs:/FileStore/shared_uploads/kumar@vcloudmatesolutions.com/dep.csv','/mydir/raw/dep.csv')

In [0]:
display(dbutils.fs.ls('/mydir/raw/'))

In [0]:
bronze_empdf = spark.read.format('csv') \
                         .option('header',True)\
                         .option('InferSchema',True)\
                         .load('dbfs:/mydir/raw/emp.csv')

In [0]:
display(bronze_empdf)

emp_id,emp_name,department_id,salary
1,John Doe,101,50000
2,Jane Smith,102,60000
3,Michael Johnson,101,55000
4,Emily Davis,103,48000
5,Robert Brown,102,62000
6,Emma Wilson,101,51000
7,William Jones,103,49000
8,Olivia Martinez,102,63000
9,Liam Taylor,101,52000
10,Sophia Anderson,103,50000


In [0]:
bronze_depdf = spark.read.format('csv') \
                         .option('header',True)\
                         .option('InferSchema',True)\
                         .load('dbfs:/mydir/raw/dep.csv')

In [0]:
display(bronze_depdf)

department_id,department_name,location
101,Engineering,New York
102,Marketing,San Francisco
103,Finance,Chicago
104,Human Resources,Los Angeles


### Silver Layer

In [0]:
silver_empwithdepjoineddf = bronze_empdf.join(bronze_depdf,'department_id')

In [0]:
display(silver_empwithdepjoineddf)

department_id,emp_id,emp_name,salary,department_name,location
101,1,John Doe,50000,Engineering,New York
102,2,Jane Smith,60000,Marketing,San Francisco
101,3,Michael Johnson,55000,Engineering,New York
103,4,Emily Davis,48000,Finance,Chicago
102,5,Robert Brown,62000,Marketing,San Francisco
101,6,Emma Wilson,51000,Engineering,New York
103,7,William Jones,49000,Finance,Chicago
102,8,Olivia Martinez,63000,Marketing,San Francisco
101,9,Liam Taylor,52000,Engineering,New York
103,10,Sophia Anderson,50000,Finance,Chicago


In [0]:
#dbutils.fs.rm('/mydir/silver',True)
#dbutils.fs.ls('/mydir/silver')

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-1702061406773642>, line 2
      1 #dbutils.fs.rm('/mydir/silver',True)
----> 2 dbutils.fs.ls('/mydir/silver')

File /databricks/python_shell/lib/dbruntime/dbutils.py:158, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
    156 exc.__context__ = None
    157 exc.__cause__ = None
--> 158 raise exc

ExecutionError: An error occurred while calling o422.ls.
: java.io.FileNotFoundException: /mydir/silver
	at com.databricks.backend.daemon.data.client.DbfsClient.send0(DbfsClient.scala:126)
	at com.databricks.backend.daemon.data.client.DbfsClient.sendIdempotent(DbfsClient.scala:74)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystemV1.listStatus(DatabricksFileSystemV1.scala:179)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.listStatus(DatabricksFileSystem.scala:1

In [0]:
dbutils.fs.mkdirs('/mydir/silver/')

True

In [0]:
silver_empwithdepjoineddf.write.format('delta')\
                                .save('/mydir/silver/')

In [0]:
df = spark.read.format('delta') \
                                .load('/mydir/silver/')

In [0]:
display(df)

department_id,emp_id,emp_name,salary,department_name,location
101,1,John Doe,50000,Engineering,New York
102,2,Jane Smith,60000,Marketing,San Francisco
101,3,Michael Johnson,55000,Engineering,New York
103,4,Emily Davis,48000,Finance,Chicago
102,5,Robert Brown,62000,Marketing,San Francisco
101,6,Emma Wilson,51000,Engineering,New York
103,7,William Jones,49000,Finance,Chicago
102,8,Olivia Martinez,63000,Marketing,San Francisco
101,9,Liam Taylor,52000,Engineering,New York
103,10,Sophia Anderson,50000,Finance,Chicago


### Gold Layer

In [0]:
gold_empwithdepdf = spark.read.format('delta') \
                              .load('/mydir/silver/')

In [0]:
gold_empwithdepdf.write.format('delta').saveAsTable('empwithdep_tbl2')

In [0]:
display(spark.sql('SELECT AVG(salary) as AVGSalaryByDep,department_name FROM empwithdep_tbl2 GROUP BY department_name'))

AVGSalaryByDep,department_name
52000.0,Engineering
49000.0,Finance
61666.666666666664,Marketing


Databricks visualization. Run in Databricks to view.